# Relatedness

In this exercise you will run a complete relatedness estimation pipeline on a real SNP dataset. By the end you should be able to:

- Prepare genomic data for relatedness analysis (LD pruning, filtering)
- Estimate pairwise kinship with KING-robust (PLINK2)
- Compute a 2D site frequency spectrum with ANGSD using genotype likelihoods
- Run relateAdmix for admixture-aware relatedness
- Visualise and compare results in R

<img src="https://natur.gl/wp-content/uploads/2021/04/RensdyrTyre_CEgevang-2048x1365.jpg" alt="image info" />

In [ ]:
### make directory for the exercise and move into it
mkdir -p ~/kenya2026/relatedness
cd ~/kenya2026/relatedness

We will be using the data set of called genotypes from different reindeer populations in Greenland (cold!!).

Here is a map from Greenland with the different reindeer populations:

<img src="https://natur.gl/wp-content/uploads/2021/04/RensdyrBestande-1280x2106.png" alt="image info" />

## STEP 1 — Data Preparation

### 1a. Inspect the Reindeer dataset

In [ ]:
# How many individuals and SNPs?
wc -l /course/kenya2026/nuno/relatedness/Reindeer.fam
wc -l /course/kenya2026/nuno/relatedness/Reindeer.bim
  
# Population breakdown
cut -f2 /course/kenya2026/nuno/relatedness/Reindeer.fam | sort | uniq -c

**How many individuals?**

**And how many SNPS?**

**How many populations?**

**And how many individuals per population?**

The next command takes a while to run so go for a walk and run the next command in 5 minutes

In [ ]:
PCAone -b /course/kenya2026/nuno/relatedness/Reindeer -k 6 --ld -o pcaone

In [ ]:
PCAone -B pcaone.residuals --match-bim pcaone.mbim --ld-r2 0.1 --ld-bp 1000000 -o pcaone

In [ ]:
echo ----Apply pruning----
plink2 --bfile /course/kenya2026/nuno/relatedness/Reindeer \
  --extract pcaone.ld.prune.in \
  --maf 0.05 \
  --geno 0.05 \
  --make-bed \
  --out Reindeer_pruned \
  --threads 4
echo ""
echo ---- Count SNPs----
echo "SNPs before pruning: $(wc -l < /course/kenya2026/nuno/relatedness/Reindeer.bim)"
echo "SNPs after pruning:  $(wc -l < Reindeer_pruned.bim)"

**Suggestion**: Always run LD pruning before KING/relatedness analysis, but **NOT** before FST or SFS estimation.
LD-pruned data biases allele frequency estimates.

**Which filters did we use?**

Use the plink site to understand better:
[Plink2 filters](https://www.cog-genomics.org/plink/2.0/filter)

**What fraction of SNPs were retained after pruning? Is this expected?**

## STEP 2 — KING Kinship Estimation

PLINK2 implements the KING-robust estimator directly. It computes pairwise kinship for all sample pairs.

In [ ]:
echo ----Calculate kinship coefficients using KING----
plink2 --bfile Reindeer_pruned \
  --make-king-table \
  --out Reindeer_king \
  --threads 4

Output file: kinship_results.kin0 Columns: #IID1 IID2 NSNP HETHET IBS0 KINSHIP

In [ ]:
echo ---- Quick summary----
head Reindeer_king.kin0
echo ""
echo ---- Count pairs by degree----
awk 'NR>1 {
  if ($8 > 0.354)      print "Duplicate/MZ_twin"
  else if ($8 > 0.177) print "1st_degree"
  else if ($8 > 0.088) print "2nd_degree"
  else if ($8 > 0.044) print "3rd_degree"
  else                 print "Unrelated"
}' Reindeer_king.kin0 | sort | uniq -c | sort -rn

The values in the pair counts are huge ! **Can you think why?** 

**Are there any close relatives in this dataset? Which degree?**

In [ ]:
source("/course/kenya2026/nuno/relatedness/plot_pca_king.R")

plot_king_ibs0(
  king_file = "~/kenya2026/relatedness/Reindeer_king.kin0"
)

In [ ]:
output_file <- file.path(getwd(), "king_heatmap_final.png")
cat("Output:", output_file, "\n")

cat("Device dimensions:", dev.size("px"), "\n")

source("/course/kenya2026/nuno/relatedness/plot_king_heatmap.R")

plot_king_heatmap(
  king_file = "~/kenya2026/relatedness/Reindeer_king.kin0"
)

**Does the first graph look familiar?** Remember the presentation

**And how do you interprete the red values in the heatmap? And why do they seem to group in specific squares?**


## STEP 3 — Remove the related individuals

Considering we have related individuals in the dataset, we should remove them. The stardart is to remove 1st degree or closer

In [ ]:
# Extract pairs with kinship > 0.177 (1st degree or closer)
awk 'NR>1 && $6 > 0.177 {print $1"\n"$2}' Reindeer_king.kin0 \
  | sort -u > relatives_to_flag.txt

echo "Individuals to consider removing: $(wc -l < relatives_to_flag.txt)"

Now we should see if removing the related individuals has an impact in our results:

In [ ]:
echo ---- Calculate the pca with all individuals -----
plink2 \
  --bfile Reindeer_pruned \
  --pca 10 \
  --out pca_Wrelated
echo""
echo ---- Calculate the pca after removing related individuals----
plink2 \
  --bfile Reindeer_pruned \
  --king-cutoff-table Reindeer_king.kin0 0.177 \
  --pca 10 \
  --out pca_Nonrelated

In [ ]:
options(
    repr.plot.width = 11,
    repr.plot.height = 8,
    repr.plot.res = 120
  )

source("/course/kenya2026/nuno/relatedness/plot_pca_king.R")

plot_pca_before_after(
    before_eigenvec = "~/kenya2026/relatedness/pca_Wrelated.eigenvec",
    after_eigenvec  = "~/kenya2026/relatedness/pca_Nonrelated.eigenvec",
    population_file = "/course/kenya2026/nuno/relatedness/population.tsv",
  )

**Do you see any differences? Is it important to remove related individuals?**

**Important**: In a real analysis you would remove one individual from each related pair (keeping the one with more data/higher coverage) before running ADMIXTURE, PCA, or FST.

# $F_{st}$ in wildebeest

We are back to wildebeest!

In this exercise we will cover:
 - Generating and displaying pairwise $F_{st}$ values
    
    
Tools used: plink2, R

The notebooks are editable, so feel free to experiment and change the code to see what happens or write notes in the text cells. Just remember to download the notebooks used here at some point if you want to save them with your own changes included.

In [ ]:
### make directory for the exercise
mkdir -p ~/kenya2026/Fst
cd ~/kenya2026/Fst

We will be using the data set of called genotypes from different blue wildebeest populations, as well as some black wildebeest as an outgroup to compare to, saved in a plink format file set. 

Here is the map from earlier to help show the sampling locations of the different wildebeest populations:
<img src="https://raw.githubusercontent.com/popgenDK/popgenDK.github.io/gh-pages/images/slider/wildeBeastMap.png" alt="image info" />


 **- Do you remember what a plink file set (.bed, bim and .fam) contains?**

In [ ]:
head /davidData/users/thomas/workshop/wildebeest_fst.fam

In [ ]:
head /davidData/users/thomas/workshop/wildebeest_fst.bim

Below we have the command used to run the $F_{st}$ estimation:

In [ ]:
plink2 --bfile /davidData/users/thomas/workshop/wildebeest_fst --within /davidData/users/thomas/workshop/clusterfile \
    --fst CATPHENO method=hudson --allow-extra-chr --threads 10

 **- How many individuals are in this files? And divided in how many populations?**

The cluster/within file give with `-within /davidData/users/thomas/workshop/clusterfile` tells the program how to separate the individuals into different groups for comparison. If we did not know up front which samples belonged together in populations, can you recall something we have looked at that could perhaps help with this?

Then let's have a look at the results:

In [ ]:
# some hartebeest samples were also originally included in this data set, but now we can just remove those from
# the output
grep -ve Hartebeest plink2.fst.summary > tmp
mv tmp plink2.fst.summary

# print the results
column -t plink2.fst.summary

 **- Which populations are most genetically differentiated? Which are most similar?**
 
 **- Can you indentify a pattern in the Fst values between black wildebeest and each of the blue wildebeest populations? Try to see if you can explain this pattern.**

Are each of these values large or small? This is quite difficult to answer without context, as it will depend on the type of data you are analyzing, the amount of data and the scope of your study. To provide context, one often looks at a matrix of $F_{st}$ values, which can be visualized using a heatmap. To do this we first need to transform the above data frame into a matrix, and then generate a heatmap using the heatmap.2-function.

In [ ]:
options(repr.matrix.max.cols=10, repr.matrix.max.rows=10)
options(repr.plot.width=16, repr.plot.height=16)
library(gplots)

# read the data into R
fst <- read.table("~/kenya2026/Fst/plink2.fst.summary")
names(fst) <- c("pop1", "pop2", "est")
fst <- fst[fst$pop1 != "Hartebeest" & fst$pop2 != "Hartebeest",]

Here we transform the table from above into a pairwise matrix that contains the exact same information, just in a different format:

In [ ]:
mat <- matrix(NA, 8, 8)
mat[lower.tri(mat)] <- fst$est
mat <- t(mat)
mat[lower.tri(mat)] <- fst$est
colnames(mat) <- c( "Amboseli", fst[1:7,2])
rownames(mat) <- c( "Amboseli", fst[1:7,2])
mat

In [ ]:
heatmap.2(mat, symm=T, trace='n', cexRow=1.5, cexCol=1.5, margins = c(12, 12))

**- Look at the clustering tree produced by this method. Do the different groups relate to each other as we would expect?**

**- We can see some discrete levels of values in the color key and in the histogram in the inset plot. What do these correspond to?**
 
An important note here is that the tree/dendrogram used to order the groups here simply comes from clustering based on the $F_{st}$ values and will not neccesarily reflect the true evolutionary history of the groups.